# ✍️ AuraDB — OpenAI RAG (Ejercicio de **consultas**) con LangChain
Cuaderno de **alumno** para consultar un grafo **ya creado** en *Neo4j AuraDB* (IMDB).  
Usaremos **LangChain + OpenAI** para NL→Cypher y responder en español.

**Estructura de cada sección:**  
- Explicación + **Pistas clave**  
- Celda de **código con TODOs** (tú la completas)  
- **### Solución (mostrar/ocultar)** y una celda con la **solución ejecutable**

## ▶️ Instalación rápida
**Docs útiles**
- AuraDB (conectar apps): https://neo4j.com/docs/aura/connecting-applications/overview/  
- Neo4j Python Driver: https://neo4j.com/docs/python-manual/current/  
- LangChain (Neo4j): https://python.langchain.com/docs/integrations/graphs/neo4j  
- Prompt templates: https://python.langchain.com/docs/guides/prompt_templates/

# Recomendado: ejecutar esta celda primero (reinicia kernel si actualiza mucho)
%pip install -q python-dotenv neo4j langchain langchain-community langchain-openai langchain-neo4j tiktoken

## 1) Configuración inicial (API Keys + modelo + conexión Neo4j)
**Objetivo:** preparar imports, cargar `.env`, configurar **OpenAI** y crear el conector `Neo4jGraph` (solo lectura).

**Pistas clave**
- `from dotenv import load_dotenv`; `load_dotenv()`  
- Variables: `OPENAI_API_KEY`, `OPENAI_MODEL` (p. ej. `gpt-4o-mini`)  
- `from langchain_openai import ChatOpenAI` + `os.environ["OPENAI_API_KEY"] = ...`  
- `from langchain_neo4j import Neo4jGraph` y `Neo4jGraph(url=..., username=..., password=...)`  
- Prueba: `graph.query("RETURN 1 AS ok")`

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or "<tu-openai-key>"
OPENAI_MODEL   = os.getenv("OPENAI_MODEL")   or "gpt-4o-mini"
NEO4J_URI      = os.getenv("NEO4J_URI")      or "neo4j+s://<tu-host>.databases.neo4j.io"
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME") or "neo4j"
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD") or "<tu-contraseña>"

from langchain_openai import ChatOpenAI
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)

from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)
graph.query("RETURN 1 AS ok")

[{'ok': 1}]

## 2) Esquema del grafo (solo lectura)
**Objetivo:** refrescar e inspeccionar el esquema para guiar la generación de Cypher.

**Pistas clave**
- `graph.refresh_schema()`  
- `print(graph.schema[:1000])`

In [2]:
# Refresca el esquema del grafo y lo imprime parcialmente para inspección.
graph.refresh_schema()  
# Imprime los primeros 1000 caracteres del esquema del grafo.
# Si el esquema es más largo, añade "..." al final para indicar truncamiento.
print(graph.schema[:1000] + ("..." if len(graph.schema) > 1000 else ""))

Node properties:
Movie {movieId: STRING, runtimeMin: INTEGER, grossUSD: FLOAT, year: INTEGER, description: STRING, imdbRating: FLOAT, votes: INTEGER, metascore: INTEGER, title: STRING}
Year {value: INTEGER}
Decade {value: INTEGER}
RatingBand {name: STRING, min: FLOAT, max: FLOAT}
RuntimeBand {name: STRING, min: INTEGER, max: INTEGER}
BoxOfficeBand {name: STRING, min: FLOAT, max: FLOAT}
Keyword {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:RELEASED_IN]->(:Year)
(:Movie)-[:HAS_RATING_BAND]->(:RatingBand)
(:Movie)-[:HAS_RUNTIME_BAND]->(:RuntimeBand)
(:Movie)-[:HAS_BOXOFFICE_BAND]->(:BoxOfficeBand)
(:Movie)-[:HAS_KEYWORD]->(:Keyword)
(:Year)-[:IN_DECADE]->(:Decade)


## 3) Prompt de generación de Cypher (solo lectura)
**Objetivo:** construir un `PromptTemplate` que **prohíba escrituras** y use `{schema}` y `{question}`.

**Pistas clave**
- `from langchain.prompts import PromptTemplate`  
- `input_variables=["schema","question"]`  
- Incluir: “NO usar CREATE/MERGE/SET/DELETE/REMOVE/DROP/LOAD CSV”

In [3]:
from langchain_core.prompts import PromptTemplate

CYTHER_PROMPT_TMPL = PromptTemplate(
    input_variables=["schema","question"],
    template=(
        "Eres un generador de Cypher para Neo4j. "
        "Responde SOLO con una consulta de LECTURA. "
        "NO crear/modificar datos (prohibido CREATE, MERGE, SET, DELETE, REMOVE, DROP, LOAD CSV).\n"
        "Esquema:\n{schema}\n\n"
        "Pregunta: {question}\n"
        "Cypher:"
    ),
)

## 4) Utilidad `clean_fences(text)`
**Objetivo:** si el LLM devuelve ```cypher ...```, quitar fences y etiquetas para quedarnos solo con el Cypher.

**Pistas clave**
- `text.strip()` y `splitlines()`  
- Si empieza por ```, quitar backticks y primera línea si no empieza por `MATCH/CALL/RETURN/WITH`

In [4]:
def clean_fences(text: str) -> str:
    """
    Limpia el texto generado por el LLM eliminando etiquetas de formato y líneas innecesarias.

    Args:
        text (str): Texto que puede contener etiquetas de formato como ```cypher.

    Returns:
        str: Texto limpio, listo para ser interpretado como una consulta Cypher.
    """
    if not text:
        # Si el texto está vacío, se devuelve tal cual.
        return text
    text = text.strip()  # Elimina espacios en blanco al inicio y al final.
    if text.startswith("```"):
        # Si el texto comienza con ``` (etiquetas de bloque de código), las elimina.
        text = text.strip("`")
        lines = text.splitlines()  # Divide el texto en líneas.
        if lines and not lines[0].strip().upper().startswith(("MATCH", "CALL", "RETURN", "WITH")):
            # Si la primera línea no contiene una palabra clave Cypher válida, se elimina.
            lines = lines[1:]
        # Une las líneas restantes en un solo string, eliminando espacios innecesarios.
        text = "\n".join(lines).strip()
    return text

## 5) Validador: **solo lectura**
**Objetivo:** rechazar Cypher con escrituras (CREATE/MERGE/SET/DELETE/REMOVE/DROP/LOAD CSV / CALL dbms...).

**Pistas clave**
- `import re` + `re.compile(r"...", re.IGNORECASE)`  
- Función `is_read_only(cy: str) -> bool` que busca con `.search(...)`

In [5]:
import re

# Define un patrón de expresiones regulares para detectar comandos Cypher que modifican datos.
# Estos incluyen CREATE, MERGE, SET, DELETE, REMOVE, DROP, LOAD CSV y llamadas específicas como CALL dbms.
READ_ONLY_PATTERN = re.compile(
    r"\b(CREATE|MERGE|SET|DELETE|REMOVE|DROP|LOAD\s+CSV|CALL\s+dbms)\b", 
    re.IGNORECASE  # Ignora mayúsculas y minúsculas al buscar coincidencias.
)

def is_read_only(cy: str) -> bool:
    """
    Verifica si una consulta Cypher es de solo lectura.

    Args:
        cy (str): Consulta Cypher a validar.

    Returns:
        bool: True si la consulta es de solo lectura, False si contiene comandos de escritura.
    """
    # Busca coincidencias con el patrón de escritura en la consulta Cypher.
    # Si no hay coincidencias, la consulta es de solo lectura.
    return not bool(READ_ONLY_PATTERN.search(cy or ""))

## 6) Función `generate_cypher(llm, schema, question)`
**Objetivo:** construir el prompt, invocar el LLM, limpiar fences y validar que sea **solo lectura**.

**Pistas clave**
- `prompt = CYTHER_PROMPT_TMPL.format(schema=schema, question=question)`  
- `text = llm.invoke(prompt).content`  
- `cy = clean_fences(text)` y `is_read_only(cy)`

In [6]:
def generate_cypher(llm, schema: str, question: str) -> str:
    """
    Genera una consulta Cypher basada en una pregunta en lenguaje natural y el esquema del grafo.

    Args:
        llm: Modelo de lenguaje utilizado para generar la consulta.
        schema (str): Esquema del grafo que sirve como contexto para la generación.
        question (str): Pregunta en lenguaje natural que debe responderse con Cypher.

    Returns:
        str: Consulta Cypher generada, validada como de solo lectura.

    Raises:
        ValueError: Si la consulta generada contiene comandos de escritura.
    """
    # Formatea el prompt utilizando el esquema y la pregunta proporcionados.
    prompt = CYTHER_PROMPT_TMPL.format(schema=schema, question=question)
    # Invoca el modelo de lenguaje para generar la consulta Cypher.
    text = llm.invoke(prompt).content
    # Limpia el texto generado para eliminar etiquetas de formato y líneas innecesarias.
    cy = clean_fences(text)
    # Valida que la consulta sea de solo lectura.
    if not is_read_only(cy):
        # Lanza un error si la consulta contiene comandos de escritura.
        raise ValueError(f"Consulta no segura (posible escritura):\n{cy}")
    # Devuelve la consulta Cypher validada.
    return cy

## 7) Función `run_cypher(graph, cypher)`
**Objetivo:** ejecutar la consulta de solo lectura y devolver registros (lista de dicts).

**Pistas clave**
- `graph.query(cypher, params or {})`

In [7]:
def run_cypher(graph, cypher: str, params=None):
    """
    Ejecuta una consulta Cypher en el grafo Neo4j y devuelve los resultados.

    Args:
        graph: Objeto de conexión al grafo (Neo4jGraph).
        cypher (str): Consulta Cypher que se desea ejecutar.
        params (dict, opcional): Parámetros adicionales para la consulta Cypher.

    Returns:
        list[dict]: Lista de registros devueltos por la consulta, donde cada registro es un diccionario.
    """
    # Ejecuta la consulta Cypher en el grafo con los parámetros proporcionados (o un diccionario vacío si no hay parámetros).
    return graph.query(cypher, params or {})

## 8) Función `answer_question(question)`
**Objetivo:** generar Cypher, **imprimirlo**, ejecutarlo y resumir con el LLM en español.

**Pistas clave**
- `cy = generate_cypher(llm, graph.schema, question)`  
- `print(cy)` antes de ejecutar  
- `rows = run_cypher(graph, cy)`  
- `PromptTemplate` para redactar respuesta corta a partir de filas

In [8]:
from langchain_core.prompts import PromptTemplate as _PT

# Define un template para generar una respuesta basada en la pregunta y los datos obtenidos.
ANSWER_PROMPT_TMPL = _PT(
    input_variables=["question", "rows"],  # Variables que se inyectarán en el template.
    template=(
        "Pregunta: {question}\n"
        "Datos (JSON abreviado): {rows}\n\n"
        "Redacta una respuesta breve y clara en español, sin inventar datos."
    )
)

def answer_question(question: str, max_rows: int = 20, show_cypher: bool = False):
    """
    Genera una consulta Cypher, la ejecuta y utiliza los resultados para redactar una respuesta.

    Args:
        question (str): Pregunta en lenguaje natural que debe responderse.
        max_rows (int): Número máximo de filas a incluir en la respuesta. Por defecto, 20.
        show_cypher (bool): Si es True, imprime la consulta Cypher generada antes de ejecutarla.

    Returns:
        dict: Un diccionario con la consulta Cypher generada, los resultados obtenidos y la respuesta redactada.
    """
    # Genera la consulta Cypher basada en la pregunta y el esquema del grafo.
    cy = generate_cypher(llm, graph.schema, question)
    
    if show_cypher:
        # Imprime la consulta Cypher generada si show_cypher es True.
        print("— Cypher generado —")
        print(cy)
        print("———————")
    
    # Ejecuta la consulta Cypher y obtiene los resultados.
    rows = run_cypher(graph, cy)
    
    # Limita el número de filas a incluir en la respuesta según max_rows.
    short_rows = rows[:max_rows]
    
    # Genera una respuesta en español utilizando el template y los datos obtenidos.
    ans = llm.invoke(ANSWER_PROMPT_TMPL.format(question=question, rows=short_rows)).content
    
    # Devuelve un diccionario con la consulta Cypher, los resultados y la respuesta generada.
    return {"cypher": cy, "rows": rows, "answer": ans}

## 9) Pruebas guiadas
Ejecuta estas pruebas cuando termines los ejercicios.

In [9]:
#  — Prueba NL→Cypher
out = answer_question("Dame el top 5 de películas de la década de 1990 con mayor puntuación IMDb.")
out["answer"]

"Aquí tienes el top 5 de películas de la década de 1990 con mayor puntuación en IMDb:\n\n1. **The Shawshank Redemption** - 9.3\n2. **Schindler's List** - 9.0\n3. **Pulp Fiction** - 8.9\n4. **Forrest Gump** - 8.8\n5. **Fight Club** - 8.8"

In [10]:
#  — Prueba Cypher directo
res = graph.query("""
MATCH (m:Movie)
RETURN m.title AS title, m.imdbRating AS rating
ORDER BY rating DESC, m.votes DESC
LIMIT 5
""")
res

[{'title': 'The Shawshank Redemption', 'rating': 9.3},
 {'title': 'The Godfather', 'rating': 9.2},
 {'title': 'The Dark Knight', 'rating': 9.0},
 {'title': "Schindler's List", 'rating': 9.0},
 {'title': 'The Lord of the Rings: The Return of the King', 'rating': 9.0}]

In [11]:
#  — Prueba NL→Cypher
out = answer_question("Nombre de películas que duran más de 150 minutos?")
out["answer"]

'Las películas que duran más de 150 minutos son:\n\n1. The Lord of the Rings: The Return of the King\n2. The Godfather: Part II\n3. Jai Bhim\n4. The Lord of the Rings: The Two Towers\n5. The Lord of the Rings: The Fellowship of the Ring\n6. The Good, the Bad and the Ugly\n7. The Green Mile\n8. Seven Samurai\n9. Sardar Udham\n10. Gladiator\n11. Cinema Paradiso'

In [12]:
#  — Prueba Cypher directo
res = graph.query("""
MATCH (m:Movie)
WHERE m.runtimeMin > 150
RETURN m.title AS movieTitle
""")
res

[{'movieTitle': 'The Godfather'},
 {'movieTitle': 'The Dark Knight'},
 {'movieTitle': 'The Lord of the Rings: The Return of the King'},
 {'movieTitle': "Schindler's List"},
 {'movieTitle': 'The Godfather: Part II'},
 {'movieTitle': 'Jai Bhim'},
 {'movieTitle': 'Pulp Fiction'},
 {'movieTitle': 'The Lord of the Rings: The Two Towers'},
 {'movieTitle': 'The Lord of the Rings: The Fellowship of the Ring'},
 {'movieTitle': 'The Good, the Bad and the Ugly'},
 {'movieTitle': 'Soorarai Pottru'},
 {'movieTitle': 'Vikram'},
 {'movieTitle': 'Interstellar'},
 {'movieTitle': 'Saving Private Ryan'},
 {'movieTitle': 'The Green Mile'},
 {'movieTitle': 'Seven Samurai'},
 {'movieTitle': 'Sardar Udham'},
 {'movieTitle': 'The Departed'},
 {'movieTitle': 'Gladiator'},
 {'movieTitle': 'Cinema Paradiso'},
 {'movieTitle': 'Once Upon a Time in the West'},
 {'movieTitle': 'Avengers: Endgame'},
 {'movieTitle': 'Django Unchained'},
 {'movieTitle': 'The Dark Knight Rises'},
 {'movieTitle': 'Drishyam 2'},
 {'movieT

In [13]:
#  — Prueba NL→Cypher
out = answer_question("Qué películas fueron creadas en 2016 y cuáles son sus nombres?")
out["answer"]

'Las películas creadas en 2016 son:\n\n1. Your Name.\n2. Dangal\n3. A Silent Voice: The Movie\n4. The Handmaiden\n5. Hacksaw Ridge\n6. The Invisible Guest\n7. La La Land\n8. Lion\n9. Zootopia\n10. Deadpool\n11. Airlift\n12. M.S. Dhoni: The Untold Story\n13. Sing Street\n14. I, Daniel Blake\n15. Hidden Figures\n16. Hunt for the Wilderpeople\n17. Manchester by the Sea\n18. Rogue One: A Star Wars Story\n19. Captain Fantastic\n20. Captain America: Civil War'

In [14]:
#  — Prueba Cypher directo
res = graph.query("""
MATCH (m:Movie)-[:RELEASED_IN]->(y:Year {value: 2016})
RETURN m.title AS movieName
""")
res

[{'movieName': 'Your Name.'},
 {'movieName': 'Dangal'},
 {'movieName': 'A Silent Voice: The Movie'},
 {'movieName': 'The Handmaiden'},
 {'movieName': 'Hacksaw Ridge'},
 {'movieName': 'The Invisible Guest'},
 {'movieName': 'La La Land'},
 {'movieName': 'Lion'},
 {'movieName': 'Zootopia'},
 {'movieName': 'Deadpool'},
 {'movieName': 'Airlift'},
 {'movieName': 'M.S. Dhoni: The Untold Story'},
 {'movieName': 'Sing Street'},
 {'movieName': 'I, Daniel Blake'},
 {'movieName': 'Hidden Figures'},
 {'movieName': 'Hunt for the Wilderpeople'},
 {'movieName': 'Manchester by the Sea'},
 {'movieName': 'Rogue One: A Star Wars Story'},
 {'movieName': 'Captain Fantastic'},
 {'movieName': 'Captain America: Civil War'},
 {'movieName': 'The Salesman'},
 {'movieName': 'Perfect Strangers'},
 {'movieName': 'Udta Punjab'},
 {'movieName': 'Kubo and the Two Strings'}]

In [15]:
#  — Prueba NL→Cypher
out = answer_question("¿Cuál es el año con más películas estrenadas, si empatan dame el año más reciente?")
out["answer"]

'El año con más películas estrenadas es 2014, con un total de 29 películas.'

In [16]:
#  — Prueba Cypher directo
res = graph.query("""
MATCH (:Movie)-[:RELEASED_IN]->(y:Year)
RETURN y.value AS year, count(*) AS num
ORDER BY num DESC, year DESC
LIMIT 10;
""")
res

[{'year': 2014, 'num': 29},
 {'year': 2004, 'num': 29},
 {'year': 2009, 'num': 26},
 {'year': 2019, 'num': 25},
 {'year': 2013, 'num': 25},
 {'year': 2006, 'num': 25},
 {'year': 2001, 'num': 25},
 {'year': 2016, 'num': 24},
 {'year': 2007, 'num': 24},
 {'year': 2012, 'num': 23}]

## 📚 Ejemplos adicionales de pruebas

A continuación se presentan más ejemplos de pruebas con `answer_question` y su validación correspondiente con consultas Cypher directas.

In [17]:
#  — Prueba NL→Cypher: Películas con más ingresos
out = answer_question("¿Cuáles son las 5 películas con mayores ingresos en taquilla?")
print(out["answer"])

No se pueden determinar las 5 películas con mayores ingresos en taquilla a partir de los datos proporcionados, ya que todos los valores de ingresos están marcados como "None".


In [18]:
#  — Prueba Cypher directo: Películas con más ingresos
res = graph.query("""
MATCH (m:Movie)
WHERE m.grossUSD IS NOT NULL
RETURN m.title AS title, m.grossUSD AS gross
ORDER BY gross DESC
LIMIT 5
""")
res

[{'title': 'Star Wars: Episode VII - The Force Awakens', 'gross': 936660000.0},
 {'title': 'Avengers: Endgame', 'gross': 858370000.0},
 {'title': 'Spider-Man: No Way Home', 'gross': 804750000.0},
 {'title': 'Avatar', 'gross': 760510000.0},
 {'title': 'Avengers: Infinity War', 'gross': 678820000.0}]

In [19]:
#  — Prueba NL→Cypher: Películas de acción
out = answer_question("¿Qué películas tienen la palabra 'Action' como keyword?")
print(out["answer"])

No hay películas en la base de datos que contengan la palabra 'Action' como keyword.


In [20]:
#  — Prueba Cypher directo: Películas con keyword Action
res = graph.query("""
MATCH (m:Movie)-[:HAS_KEYWORD]->(k:Keyword {name: 'Action'})
RETURN m.title AS title, m.year AS year
ORDER BY year DESC
LIMIT 10
""")
res

[]

In [21]:
#  — Prueba NL→Cypher: Películas por banda de rating
out = answer_question("¿Cuántas películas hay en cada banda de rating?")
print(out["answer"])

En las diferentes bandas de rating hay la siguiente cantidad de películas:

- Menos de 8.0: 515 películas
- Entre 8.0 y 8.5: 392 películas
- Entre 8.5 y 9.0: 53 películas
- Entre 9.0 y 10: 7 películas


In [22]:
#  — Prueba Cypher directo: Películas por banda de rating
res = graph.query("""
MATCH (m:Movie)-[:HAS_RATING_BAND]->(rb:RatingBand)
RETURN rb.name AS ratingBand, count(m) AS numMovies
ORDER BY numMovies DESC
""")
res

[{'ratingBand': '< 8.0', 'numMovies': 515},
 {'ratingBand': '[8.0–8.5)', 'numMovies': 392},
 {'ratingBand': '[8.5–9.0)', 'numMovies': 53},
 {'ratingBand': '[9.0–10)', 'numMovies': 7}]

In [23]:
#  — Prueba NL→Cypher: Películas por década
out = answer_question("¿Qué década tiene más películas estrenadas?")
print(out["answer"])

La década con más películas estrenadas es la de 2000, con un total de 221 películas.


In [24]:
#  — Prueba Cypher directo: Películas por década
res = graph.query("""
MATCH (m:Movie)-[:RELEASED_IN]->(y:Year)-[:IN_DECADE]->(d:Decade)
RETURN d.value AS decade, count(m) AS numMovies
ORDER BY numMovies DESC
LIMIT 10
""")
res

[{'decade': 2000, 'numMovies': 221},
 {'decade': 2010, 'numMovies': 212},
 {'decade': 1990, 'numMovies': 143},
 {'decade': 1980, 'numMovies': 89},
 {'decade': 1960, 'numMovies': 74},
 {'decade': 1970, 'numMovies': 69},
 {'decade': 1950, 'numMovies': 60},
 {'decade': 1940, 'numMovies': 38},
 {'decade': 1930, 'numMovies': 26},
 {'decade': 2020, 'numMovies': 24}]

In [25]:
#  — Prueba NL→Cypher: Películas con filtros múltiples
out = answer_question("¿Qué películas estrenadas después del 2000 tienen rating superior a 8.5 y más de 100000 votos?")
print(out["answer"])

No hay datos disponibles sobre películas estrenadas después del 2000 que tengan un rating superior a 8.5 y más de 100,000 votos.


In [26]:
#  — Prueba Cypher directo: Películas con filtros múltiples
res = graph.query("""
MATCH (m:Movie)-[:RELEASED_IN]->(y:Year)
WHERE y.value > 2000 AND m.imdbRating > 8.5 AND m.votes > 100000
RETURN m.title AS title, m.year AS year, m.imdbRating AS rating, m.votes AS votes
ORDER BY rating DESC, votes DESC
LIMIT 10
""")
res

[]

In [ ]:
#  — Prueba NL→Cypher con show_cypher=True (para ver el Cypher generado)
out = answer_question("¿Qué películas duran entre 90 y 120 minutos?")
print(out["answer"])

— Cypher generado —
MATCH (m:Movie)
WHERE m.runtimeMin >= 90 AND m.runtimeMin <= 120
RETURN m.title, m.runtimeMin
———————
Las películas que duran entre 90 y 120 minutos son:

1. 12 Angry Men - 96 minutos
2. Life Is Beautiful - 116 minutos
3. The Silence of the Lambs - 118 minutos
4. Whiplash - 106 minutos
5. The Intouchables - 112 minutos
6. American History X - 119 minutos
7. The Usual Suspects - 106 minutos
8. Léon: The Professional - 110 minutos
9. Back to the Future - 116 minutos
10. Alien - 117 minutos
11. Psycho - 109 minutos
12. Rear Window - 112 minutos
13. Casablanca - 102 minutos
14. Your Name. - 106 minutos
15. Spider-Man: Into the Spider-Verse - 117 minutos
16. WALL·E - 98 minutos
17. Oldboy - 120 minutos
18. Memento - 113 minutos
19. Indiana Jones and the Raiders of the Lost Ark - 115 minutos

Estas son las películas que cumplen con el criterio de duración.


In [31]:
#  — Prueba Cypher directo: Películas entre 90 y 120 minutos
res = graph.query("""
MATCH (m:Movie)
WHERE m.runtimeMin >= 90 AND m.runtimeMin <= 120
RETURN m.title AS title, m.runtimeMin AS runtime
""")
res

[{'title': '12 Angry Men', 'runtime': 96},
 {'title': 'Life Is Beautiful', 'runtime': 116},
 {'title': 'The Silence of the Lambs', 'runtime': 118},
 {'title': 'Whiplash', 'runtime': 106},
 {'title': 'The Intouchables', 'runtime': 112},
 {'title': 'American History X', 'runtime': 119},
 {'title': 'The Usual Suspects', 'runtime': 106},
 {'title': 'Léon: The Professional', 'runtime': 110},
 {'title': 'Back to the Future', 'runtime': 116},
 {'title': 'Alien', 'runtime': 117},
 {'title': 'Psycho', 'runtime': 109},
 {'title': 'Rear Window', 'runtime': 112},
 {'title': 'Casablanca', 'runtime': 102},
 {'title': 'Your Name.', 'runtime': 106},
 {'title': 'Spider-Man: Into the Spider-Verse', 'runtime': 117},
 {'title': 'WALL·E', 'runtime': 98},
 {'title': 'Oldboy', 'runtime': 120},
 {'title': 'Memento', 'runtime': 113},
 {'title': 'Indiana Jones and the Raiders of the Lost Ark', 'runtime': 115},
 {'title': 'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb',
  'runtime': 95},
 

In [29]:
#  — Prueba Cypher directo: Películas de 2016 por presupuesto
res = graph.query("""
MATCH (m:Movie)-[:RELEASED_IN]->(y:Year {value: 2016})
WHERE m.grossUSD IS NOT NULL
RETURN m.title AS title, m.grossUSD AS budget
ORDER BY budget DESC
LIMIT 10
""")
res

[{'title': 'Rogue One: A Star Wars Story', 'budget': 532179999.99999994},
 {'title': 'Captain America: Civil War', 'budget': 408080000.0},
 {'title': 'Deadpool', 'budget': 363070000.0},
 {'title': 'Zootopia', 'budget': 341270000.0},
 {'title': 'Hidden Figures', 'budget': 169610000.0},
 {'title': 'La La Land', 'budget': 151100000.0},
 {'title': 'Hacksaw Ridge', 'budget': 67210000.0},
 {'title': 'Lion', 'budget': 51740000.0},
 {'title': 'Kubo and the Two Strings', 'budget': 48020000.0},
 {'title': 'Manchester by the Sea', 'budget': 47700000.0}]

In [30]:
out = answer_question("Todas las peliculas que se hayan creado en el año 2016 ordernados por presupuestos mas altos a menos altos y muestrame el nombre y lo costó hacerla?")
print(out["answer"])


Aquí tienes la lista de películas de 2016 ordenadas por presupuesto de mayor a menor:

1. **Rogue One: A Star Wars Story** - $532,179,999.99
2. **Captain America: Civil War** - $408,080,000.00
3. **Deadpool** - $363,070,000.00
4. **Zootopia** - $341,270,000.00
5. **Hidden Figures** - $169,610,000.00
6. **La La Land** - $151,100,000.00
7. **Hacksaw Ridge** - $67,210,000.00
8. **Lion** - $51,740,000.00
9. **Kubo and the Two Strings** - $48,020,000.00
10. **Manchester by the Sea** - $47,700,000.00
11. **Dangal** - $12,390,000.00
12. **Captain Fantastic** - $5,880,000.00
13. **Hunt for the Wilderpeople** - $5,200,000.00
14. **Your Name.** - $5,020,000.00
15. **Sing Street** - $3,240,000.00
16. **A Silent Voice: The Movie** - $N/A
17. **The Invisible Guest** - $N/A
18. **Airlift** - $N/A
19. **Perfect Strangers** - $N/A
20. **Udta Punjab** - $N/A

Las películas con costo "N/A" no tienen datos disponibles sobre su presupuesto.


## 10) Troubleshooting
- `AuthenticationError`: revisa `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD` (usa `neo4j+s://` con Aura).  
- `ServiceUnavailable`: problemas de red/URI o instancia parada.  
- **El Cypher no se ve**: aquí se imprime antes de ejecutar (`answer_question(..., show_cypher=True)`).  
- **LLM propone escritura**: tu validador la detecta y lanza error.